In [1]:
import pandas as pd
import numpy as np

!pip install catboost
!pip install imbalanced-learn -U
!pip install specificity
from imblearn.metrics import geometric_mean_score as g_mean_score

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix
)

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore",category=ConvergenceWarning)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier , BaggingClassifier, StackingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE, ADASYN

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.9 MB/s eta 0:00:00


In [2]:
df = pd.read_csv('/content/bank-additional-full.csv', sep=';', engine='python',on_bad_lines='skip')

In [3]:
df.drop('duration', axis=1, inplace=True)

In [4]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

In [5]:
df['y'] = df['y'].map({'no': 0, 'yes': 1})

In [6]:
df = pd.get_dummies(df, drop_first=True)

In [7]:
X = df.drop('y', axis=1)
y = df['y']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
smote = SMOTE(random_state=42)

In [11]:
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [12]:
models = {
    "Logistic Regression": LogisticRegression(
        C=1,
        penalty='l2',
        solver='liblinear',
        tol=1e-2,
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=20 ,
        random_state=42,
        min_samples_split=10
        ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        max_depth=20
        ),
    "Gradient Boosting (GBM)": GradientBoostingClassifier(
        n_estimators=50,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        learning_rate=0.2
        ),
    "LightGBM": LGBMClassifier(
        learning_rate=0.1,
        n_estimators=50,
        n_jobs=-1,
        random_state=42
    ),
    "CatBoost": CatBoostClassifier(
        iterations=50,
        learning_rate=0.1,
        thread_count=-1,
        verbose=0,
        random_state=42
    ),
    "Support Vector Machine": SVC(
        kernel='rbf',
        probability=False,
        cache_size=1000,
        tol=1e-2,
        max_iter=5000,
        random_state=42
    ),
    "K Nearest Neighbour": KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1,
    ),
    "Multi Layer Perceptron": MLPClassifier(
        hidden_layer_sizes=(50,),
        max_iter=100,
        activation='relu',
        solver='adam',
        early_stopping=True,
        random_state=42
    ),
     "Bagging Classifier": BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "Stacking Classifier": StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=101, max_depth=3,  random_state=42)),
            ('knn', KNeighborsClassifier(n_neighbors=5)),
            ('dct', DecisionTreeClassifier(max_depth=5))
        ],
        final_estimator=LogisticRegression(C=0.5, max_iter=100),
        cv=KFold(n_splits=5, shuffle=True, random_state=42), # Lower folds = faster
        n_jobs=-1,            # Use all CPU cores
       passthrough=False
    )
}




In [13]:

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [14]:
param_grids = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10, 100],
        "penalty": ['l1', 'l2'],
        'solver': ['lbfgs', 'liblinear'],
    },
    "Decision Tree": {
        "max_depth": [10, 20, 30, None],
        "min_samples_split": [2, 5, 10],
        "criterion": ['gini', 'entropy']
    },
    "Random Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [10, 20, None],
        "min_samples_leaf": [1, 2, 4]
    },
    "Gradient Boosting (GBM)": {
        "n_estimators": [50, 100, 150],
        "learning_rate": [0.01, 0.1, 0.2],
        "max_depth": [3, 5, 8],
        'max_features': ['sqrt', 'log2']
    },
    "XGBoost": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.1, 0.3],
        "max_depth": [3, 6, 10]
    },
    "LightGBM": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.05, 0.1],
        "num_leaves": [31, 50, 100],
        'subsample': [0.2, 0.5, 0.8],
        'colsample_bytree': [0.2, 0.5, 0.8]
    },
    "CatBoost": {
        "iterations": [50, 100, 200],
        "learning_rate": [0.01, 0.05, 0.1],
        "depth": [4, 6, 10],
        'l2_leaf_reg': [1, 3, 5]
    },
    "Support Vector Machine": {
        "C": [0.1, 1, 10],
        "kernel": ['rbf', 'poly', 'sigmoid'],
        "gamma": ['scale', 'auto']
    },
    "K Nearest Neighbour": {
        "n_neighbors": [3, 5, 11, 21],
        "weights": ['uniform', 'distance'],
        "metric": ['euclidean', 'manhattan']
    },
    "Multi Layer Perceptron": {
        "hidden_layer_sizes": [(50,), (100,), (50, 50)],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate_init": [0.001, 0.01]
    },
    "Bagging Classifier": {
        "n_estimators": [10, 50, 100],
        "max_samples": [0.5, 0.8, 1.0],
        "max_features": [0.5, 0.8, 1.0]
    },
    "Stacking Classifier": {
        # 'final_estimator__' tunes the LogisticRegression judge
        "final_estimator__C": [0.1, 1.0, 10.0]
    }
}


In [15]:
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'specificity': make_scorer(recall_score, pos_label=0),
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'mcc':make_scorer( matthews_corrcoef),
    'g_mean_score': make_scorer(g_mean_score),
    'cohen_kappa_score': make_scorer(cohen_kappa_score)
}


In [16]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [17]:
result = []

In [18]:
def evaluate_models(X, y, label,scoring):
    print(f"\n===== {label} =====")

    for name, model in models.items():
        print(f"\n Tuning Model : {name}")

        search = RandomizedSearchCV(
            model,
            param_distributions=param_grids.get(name),
            n_iter=5 if name!="Stacking Classifier" else 1,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
            scoring=scoring,
            refit='roc_auc',
            n_jobs=-1,
            random_state=42,
            return_train_score=False
        )

        search.fit(X, y)

        print("Best Parameters:", search.best_params_)

        cv_res = search.cv_results_

        ac = round(cv_res['mean_test_accuracy'][search.best_index_], 4)
        pc = round(cv_res['mean_test_precision'][search.best_index_], 4)
        rec    = round(cv_res['mean_test_recall'][search.best_index_], 4)
        sp = round(cv_res['mean_test_specificity'][search.best_index_], 4)
        f1  = round(cv_res['mean_test_f1'][search.best_index_], 4)
        roc   = round(cv_res['mean_test_roc_auc'][search.best_index_], 4)
        mcc       = round(cv_res['mean_test_mcc'][search.best_index_], 4)
        g_mean     = round(cv_res['mean_test_g_mean_score'][search.best_index_], 4)
        c_kappa     = round(cv_res['mean_test_cohen_kappa_score'][search.best_index_], 4)



        print(f"Accuracy:  {ac}")
        print(f"Precision: {pc}")
        print(f"Recall:    {rec}")
        print(f"Specificity: {sp}")
        print(f"F1 Score:  {f1}")
        print(f"ROC-AUC:   {roc}")
        print(f"MCC:       {mcc}")
        print(f"G-Mean:    {g_mean}")
        print(f"Cohen Kappa: {c_kappa}")

        result.append([label,name , ac , pc , rec ,sp ,f1 , roc , mcc , g_mean , c_kappa])


In [19]:
evaluate_models(X_train_sm, y_train_sm, "SMOTE",scoring=scoring)


===== SMOTE =====

 Tuning Model : Logistic Regression


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
10 fits failed out of a total of 25.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py", line 1193, in fit
    solver = _check_solver

Best Parameters: {'solver': 'liblinear', 'penalty': 'l2', 'C': 10}
Accuracy:  0.7399
Precision: 0.8034
Recall:    0.6352
Specificity: 0.8445
F1 Score:  0.7095
ROC-AUC:   0.8003
MCC:       0.4906
G-Mean:    0.7324
Cohen Kappa: 0.4798

 Tuning Model : Decision Tree
Best Parameters: {'min_samples_split': 10, 'max_depth': None, 'criterion': 'gini'}
Accuracy:  0.9088
Precision: 0.9247
Recall:    0.8901
Specificity: 0.9275
F1 Score:  0.907
ROC-AUC:   0.9345
MCC:       0.8182
G-Mean:    0.9086
Cohen Kappa: 0.8176

 Tuning Model : Random Forest
Best Parameters: {'n_estimators': 100, 'min_samples_leaf': 2, 'max_depth': None}
Accuracy:  0.9339
Precision: 0.9439
Recall:    0.9227
Specificity: 0.9452
F1 Score:  0.9332
ROC-AUC:   0.9795
MCC:       0.8681
G-Mean:    0.9339
Cohen Kappa: 0.8679

 Tuning Model : Gradient Boosting (GBM)
Best Parameters: {'n_estimators': 100, 'max_features': 'sqrt', 'max_depth': 8, 'learning_rate': 0.2}
Accuracy:  0.9375
Precision: 0.9629
Recall:    0.9101
Specificity: 0

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best Parameters: {'learning_rate': 0.05, 'l2_leaf_reg': 3, 'iterations': 200, 'depth': 4}
Accuracy:  0.9215
Precision: 0.96
Recall:    0.8796
Specificity: 0.9633
F1 Score:  0.918
ROC-AUC:   0.9681
MCC:       0.8459
G-Mean:    0.9205
Cohen Kappa: 0.8429

 Tuning Model : Support Vector Machine


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best Parameters: {'kernel': 'sigmoid', 'gamma': 'auto', 'C': 0.1}
Accuracy:  0.5521
Precision: 0.5358
Recall:    0.864
Specificity: 0.2402
F1 Score:  0.6592
ROC-AUC:   0.7015
MCC:       0.1204
G-Mean:    0.4258
Cohen Kappa: 0.1042

 Tuning Model : K Nearest Neighbour
Best Parameters: {'weights': 'distance', 'n_neighbors': 11, 'metric': 'manhattan'}
Accuracy:  0.889
Precision: 0.8553
Recall:    0.9366
Specificity: 0.8415
F1 Score:  0.8941
ROC-AUC:   0.9501
MCC:       0.7816
G-Mean:    0.8878
Cohen Kappa: 0.7781

 Tuning Model : Multi Layer Perceptron
Best Parameters: {'learning_rate_init': 0.01, 'hidden_layer_sizes': (100,), 'alpha': 0.0001}
Accuracy:  0.8799
Precision: 0.8804
Recall:    0.88
Specificity: 0.8799
F1 Score:  0.8799
ROC-AUC:   0.9481
MCC:       0.7602
G-Mean:    0.8798
Cohen Kappa: 0.7599

 Tuning Model : Bagging Classifier
Best Parameters: {'n_estimators': 100, 'max_samples': 1.0, 'max_features': 0.5}
Accuracy:  0.9409
Precision: 0.9637
Recall:    0.9163
Specificity: 0.96

In [20]:
df_result= pd.DataFrame(result , columns=["Sampling","Model" , "Accuracy" ,"Specificity" ,"Precision" , "Recall","F1 Score" , "ROC-AUC" , "MCC","G-Mean","Cohen's-Kappa-Score"])

In [21]:
df_result.to_excel("Res_rsearchCV_purbasa.xlsx" , index=False)

In [22]:
from google.colab import files
files.download("Res_rsearchCV_purbasa.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>